# Sello 7 — Prueba de Hipótesis

### ¿Una sucursal vende de verdad más que otra, o es azar?

**Pregunta de negocio del sello:** ¿Existe una diferencia real en el monto promedio por transacción entre las cuatro sucursales de Café Cordillera, o las diferencias observadas se deben al azar?

**Modelo:** `monto ~ sucursal` (ANOVA de una vía + OLS con dummies + t-test de confirmación)

> Proyecto Final Integrador · TCNT0011 Probabilidad y Estadística I · Café Cordillera
>
> Método: ANOVA de una vía (scipy) para comparar las cuatro sucursales simultáneamente, OLS con statsmodels para cuantificar la diferencia por sucursal, y t-test de Welch como confirmación entre los extremos.

## 1. Datos

Cargamos `cafe_cordillera_dataset.csv`. Calculamos `monto = precio_unitario × unidades`, que es la variable dependiente.
Las sucursales son cuatro: Cartago, Escazú, Heredia y San Pedro.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

for _p in ["cafe_cordillera_dataset.csv", "../cafe_cordillera_dataset.csv"]:
    if os.path.exists(_p):
        DATA_PATH = _p
        break

df = pd.read_csv(DATA_PATH)
df["monto"] = df["precio_unitario"] * df["unidades"]
print(f"Filas: {df.shape[0]:,} · Columnas: {df.shape[1]}")
print(df.head(3).to_string(index=False))

## 2. Planteamiento de hipótesis

Usamos **ANOVA de una vía** porque hay cuatro grupos independientes (cuatro sucursales) y queremos comparar sus medias simultáneamente. Hacer múltiples t-tests inflaría el error de Tipo I (problema de comparaciones múltiples).

- **H₀:** Las cuatro sucursales tienen el mismo monto promedio por transacción.  
  μ_Cartago = μ_Escazú = μ_Heredia = μ_San Pedro
- **H₁:** Al menos una sucursal tiene un monto promedio diferente.
- **Nivel de significancia:** α = 0.05

## 3. Estadísticos descriptivos por sucursal

In [ ]:
resumen = df.groupby("sucursal")["monto"].agg(
    n="count", media="mean", std="std", mediana="median"
).round(2)
print("Estadísticos del monto (₡) por sucursal:")
print(resumen.to_string())

## 4. ANOVA de una vía

In [ ]:
grupos = [g["monto"].values for _, g in df.groupby("sucursal")]
f_stat, p_valor = stats.f_oneway(*grupos)
print(f"Estadístico F : {f_stat:.4f}")
print(f"p-value       : {p_valor:.4e}")
print(f"α             : 0.05")
print()
if p_valor < 0.05:
    print("✅ DECISIÓN: Se rechaza H₀.")
    print("   Existe evidencia estadística de que al menos una sucursal"
          " tiene un monto promedio diferente (p < 0.05).")
else:
    print("❌ DECISIÓN: No se rechaza H₀.")

## 5. Modelo OLS con dummies de sucursal

Ajustamos una regresión OLS con variables indicadoras para cuantificar la diferencia de cada sucursal respecto a la categoría de referencia (Cartago). Esto permite obtener coeficientes, errores estándar e intervalos de confianza por sucursal.

In [ ]:
import statsmodels.api as sm

d = pd.get_dummies(df, columns=["sucursal"], drop_first=True, dtype=int)
suc_cols = [c for c in d.columns if c.startswith("sucursal_")]
X = sm.add_constant(d[suc_cols].astype(float))
modelo = sm.OLS(d["monto"].astype(float), X).fit()
print(modelo.summary().tables[1].as_text())
print(f"R² = {modelo.rsquared:.4f}  ·  R² ajustado = {modelo.rsquared_adj:.4f}")

## 6. t-test de confirmación: Escazú vs Heredia (los extremos)

In [ ]:
g_escazu  = df[df["sucursal"] == "Escazu"]["monto"]
g_heredia = df[df["sucursal"] == "Heredia"]["monto"]
t_stat, p_t = stats.ttest_ind(g_escazu, g_heredia)
print(f"Media Escazú  : ₡{g_escazu.mean():,.2f}")
print(f"Media Heredia : ₡{g_heredia.mean():,.2f}")
print(f"Diferencia    : ₡{g_escazu.mean() - g_heredia.mean():,.2f}")
print(f"t-estadístico : {t_stat:.4f}")
print(f"p-value       : {p_t:.4e}")
print(f"\nEscazú supera a Heredia en un {((g_escazu.mean()/g_heredia.mean())-1)*100:.1f}% en monto promedio.")

## 7. Visualización

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Sello 7 — Monto por transacción por sucursal", fontsize=13, fontweight="bold")

colores = ["#6B4226", "#A0522D", "#CD853F", "#DEB887"]
medias  = resumen["media"]

bars = axes[0].bar(medias.index, medias.values, color=colores, edgecolor="white", width=0.6)
axes[0].axhline(medias.mean(), color="red", linestyle="--", linewidth=1.5,
                label=f"Promedio global ₡{medias.mean():,.0f}")
axes[0].set_title("Monto promedio por transacción")
axes[0].set_ylabel("Monto (₡)")
axes[0].set_ylim(2000, 2600)
axes[0].legend()
for bar, val in zip(bars, medias.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
                 f"₡{val:,.0f}", ha="center", va="bottom", fontsize=9)

data_box = [df[df["sucursal"]==s]["monto"].values for s in medias.index]
bp = axes[1].boxplot(data_box, labels=medias.index, patch_artist=True,
                     medianprops=dict(color="red", linewidth=2))
for patch, color in zip(bp["boxes"], colores):
    patch.set_facecolor(color); patch.set_alpha(0.8)
axes[1].set_title(f"Distribución del monto (ANOVA F={f_stat:.2f}, p<0.001)")
axes[1].set_ylabel("Monto (₡)")

plt.tight_layout()
plt.show()

## 8. Conclusión (lenguaje de negocio)

El ANOVA confirma con F = 18.10 y p < 0.001 que las diferencias entre sucursales no son producto del azar. **Escazú lidera** con un monto promedio de ₡2,425 por transacción, mientras que Heredia es la más baja con ₡2,220 — una brecha de ₡205 (9.3 %) que el t-test de confirmación valida con t = 6.24 y p < 0.001. El R² bajo (≈ 0.004) indica que la sucursal explica solo una fracción de la variabilidad total del monto, por lo que existen otros factores relevantes (categoría, promoción, hora). **Recomendación:** la gerencia debería identificar qué prácticas de Escazú — mix de productos, zona demográfica, atención — se pueden replicar en Heredia para reducir la brecha.